# ⚡ Fine-Tuning NVIDIA Nemotron 70B (BCP Enterprise Edition)
This notebook fine-tunes **NVIDIA Llama-3.1 Nemotron 70B Instruct** using **Unsloth rsLoRA** on your **9,189 Behavioral Counterparty Profiling (BCP) market records**.

### Why Nemotron 70B is the ultimate choice:
1. **NVIDIA's Flagship Model**: Trained specifically for complex quantitative reasoning, chain-of-thought financial deduction, and structured adherence.
2. **Massive 70B Parameter Scale**: Unrivaled depth for microstructural order flow analysis and institutional behavior modeling.
3. **Optimized for G4 (96 GB VRAM)**: Using 4-bit quantization with 16-bit LoRA, Nemotron 70B fits in ~40 GB, leaving plenty of memory for fast batching.
4. **Response-Only Masking**: Only trains on target archetype fields, eliminating prompt memorization.

### Colab Pro Runtime Configuration:
* Set **Runtime -> Change runtime type -> GPU -> G4 (96 GB VRAM)** (or A100 80GB).

In [ ]:
# Step 1: Fast installation on A100 (No xformers compile stall)
!nvidia-smi
!pip install -q unsloth
!pip install -q "trl<0.13.0" peft accelerate bitsandbytes datasets


In [ ]:
# Step 2: Upload and Unpack Training Dataset
import os, zipfile
from google.colab import files

zip_name = "bcp_finetuning_data.zip"
if not os.path.exists(zip_name):
    print("Please upload bcp_finetuning_data.zip:")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("data")

print("Dataset extracted successfully!")
for root, dirs, filenames in os.walk("data"):
    for f in filenames:
        p = os.path.join(root, f)
        print(f"  {p} ({os.path.getsize(p):,} bytes)")


In [ ]:
# Step 3: Load NVIDIA Llama-3.1 Nemotron 70B Instruct with rsLoRA
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model_name = "unsloth/Llama-3.1-Nemotron-70B-Instruct-bnb-4bit"

print(f"Loading NVIDIA Nemotron: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None, # Auto-detects bfloat16 on G4 / A100
    load_in_4bit=True,
)

# Configure Rank-Stabilized LoRA (rsLoRA) on all linear layers
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                   # High rank for complex reasoning capacity
    lora_alpha=64,          # Scaling factor
    use_rslora=True,        # Rank-Stabilized LoRA
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("NVIDIA Nemotron 70B with rsLoRA successfully initialized!")


In [ ]:
# Step 4: Tokenize Dataset with Llama 3.1 / Nemotron Chat Template
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

dataset = load_dataset("json", data_files={
    "train": "data/chatml/train.jsonl",
    "val": "data/chatml/val.jsonl",
    "test": "data/chatml/test.jsonl"
})

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Dataset mapped: Train={len(dataset['train']):,} | Val={len(dataset['val']):,} | Test={len(dataset['test']):,}")


In [ ]:
# Step 5: Train SFT on Nemotron 70B with Response-Only Masking
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

training_args = TrainingArguments(
    output_dir="bcp_nemotron70b_checkpoints",
    per_device_train_batch_size=2,      # Optimal for A100 (40GB or 80GB)
    gradient_accumulation_steps=8,      # Effective batch size = 16
    warmup_ratio=0.06,
    num_train_epochs=2,                 # 2 full passes for domain mastery
    learning_rate=1.0e-4,               # Optimal for 70B parameter LoRA
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

# Response-only masking: Calculate loss STRICTLY on target profile outputs
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

print("Launching SFT training for NVIDIA Nemotron 70B...")
trainer_stats = trainer.train()
print("Nemotron 70B Training successfully finished!")


In [ ]:
# Step 6: Test Inference on Unseen Out-of-Time Test Sample
FastLanguageModel.for_inference(model)
import re

test_subset = dataset["test"].select(range(min(10, len(dataset["test"]))))
print("Evaluating Nemotron 70B on Out-of-Time Test Records (Aug–Oct 2026)...\n")

for idx, row in enumerate(test_subset):
    msgs = row["messages"]
    prompt = msgs[:2]
    expected = msgs[2]["content"]
    
    inputs = tokenizer.apply_chat_template(
        prompt,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        out = model.generate(input_ids=inputs, max_new_tokens=280, temperature=0.1)
    predicted = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    
    print(f"--- TEST CASE {idx+1} ---")
    print("NEMOTRON PREDICTED PROFILE:\n", predicted)
    print("\nGROUND TRUTH:\n", expected)
    print("-" * 50 + "\n")


In [ ]:
# Step 7: Export Model (LoRA Adapter & 16-bit Standalone Weights)
import shutil
from google.colab import files

lora_output_dir = "bcp_nemotron70b_lora_final"
print(f"Saving final Nemotron LoRA adapter to {lora_output_dir}...")
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)

# Package LoRA adapter for immediate download
shutil.make_archive(lora_output_dir, "zip", lora_output_dir)
print(f"Created {lora_output_dir}.zip! Triggering download...")
files.download(f"{lora_output_dir}.zip")

# Optional: Save 16-bit standalone merged model for vLLM or Hugging Face Hub
# To export a merged standalone model, uncomment below:
# model.save_pretrained_merged("bcp_nemotron70b_merged_16bit", tokenizer, save_method="merged_16bit")
print("Export completed!")
